In [65]:
import os
import nibabel as nib
import numpy as np

In [66]:
IXI_T1 = r"C:\Users\Rishabh\Documents\IXI-T1"
IXI_T1 = r"C:\Users\Rishabh\Documents\PreprocessedData1mm"
Files = os.listdir(IXI_T1)

In [67]:
Files

['IXI019-Guys-0702-T1.nii.gz',
 'IXI069-Guys-0769-T1.nii.gz',
 'IXI110-Guys-0733-T1.nii.gz',
 'IXI150-HH-1550-T1.nii.gz',
 'IXI160-HH-1637-T1.nii.gz',
 'IXI163-HH-1621-T1.nii.gz',
 'IXI167-HH-1569-T1.nii.gz',
 'IXI236-Guys-0817-T1.nii.gz',
 'IXI265-Guys-0845-T1.nii.gz',
 'IXI327-HH-1999-T1.nii.gz',
 'IXI336-Guys-0904-T1.nii.gz',
 'IXI363-HH-2050-T1.nii.gz',
 'IXI399-Guys-0966-T1.nii.gz',
 'IXI414-Guys-0957-T1.nii.gz',
 'IXI429-Guys-0997-T1.nii.gz',
 'IXI436-HH-2153-T1.nii.gz',
 'IXI477-IOP-1141-T1.nii.gz']

In [98]:
ID_To_Filename = {}
for file in Files:
    strc = file.split('-')
    ID_To_Filename[int(strc[0][-3:])] = file

In [99]:
ID_To_Filename

{19: 'IXI019-Guys-0702-T1.nii.gz',
 69: 'IXI069-Guys-0769-T1.nii.gz',
 110: 'IXI110-Guys-0733-T1.nii.gz',
 150: 'IXI150-HH-1550-T1.nii.gz',
 160: 'IXI160-HH-1637-T1.nii.gz',
 163: 'IXI163-HH-1621-T1.nii.gz',
 167: 'IXI167-HH-1569-T1.nii.gz',
 236: 'IXI236-Guys-0817-T1.nii.gz',
 265: 'IXI265-Guys-0845-T1.nii.gz',
 327: 'IXI327-HH-1999-T1.nii.gz',
 336: 'IXI336-Guys-0904-T1.nii.gz',
 363: 'IXI363-HH-2050-T1.nii.gz',
 399: 'IXI399-Guys-0966-T1.nii.gz',
 414: 'IXI414-Guys-0957-T1.nii.gz',
 429: 'IXI429-Guys-0997-T1.nii.gz',
 436: 'IXI436-HH-2153-T1.nii.gz',
 477: 'IXI477-IOP-1141-T1.nii.gz'}

In [68]:
for i in range(len(Files)):
    filename = Files[i]
    strc = filename.split('-')
    Modality = strc[3][:-7]
    # print(strc)
    if strc[1] == 'Guys':
        file_path = os.path.join(IXI_T1, filename)
        img = nib.load(file_path)
        data = img.get_fdata()
        print(data.shape)
        
        print("min:- ", np.min(data), '  max:- ', np.max(data),' unique values:- ', len(np.unique(data)) )
        mean_of_array = data.mean()
        std_of_array = data.std()
        print('mean:- ', mean_of_array,' std:- ', std_of_array)
        print("--------"*10)
        

(182, 218, 182)
min:-  0.0   max:-  2381.4114607695656  unique values:-  2154593
mean:-  232.1571170298481  std:-  392.2859212299444
--------------------------------------------------------------------------------
(182, 218, 182)
min:-  0.0   max:-  1634.5997896897563  unique values:-  2206857
mean:-  280.00594577858743  std:-  464.4949784649749
--------------------------------------------------------------------------------
(182, 218, 182)
min:-  0.0   max:-  2093.56716764159  unique values:-  2189858
mean:-  258.489467560984  std:-  431.6137644910939
--------------------------------------------------------------------------------
(182, 218, 182)
min:-  0.0   max:-  765.041517525114  unique values:-  2168079
mean:-  92.25453919701881  std:-  153.44563440634377
--------------------------------------------------------------------------------
(182, 218, 182)
min:-  0.0   max:-  532.2465351377693  unique values:-  2185491
mean:-  91.37021523403074  std:-  150.2455645352847
---------------

KeyboardInterrupt: 

In [69]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

def resample_to_128(input_path):
    img = sitk.ReadImage(input_path)
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize([128, 128, 128])
    # resampler.SetOutputSpacing([1.0, 1.0, 1.0])
    # resampler.SetOutputOrigin(img.GetOrigin())
    # resampler.SetOutputDirection(img.GetDirection())
    # resampler.SetInterpolator(sitk.sitkBSpline)
    out = resampler.Execute(img)
    array = sitk.GetArrayFromImage(out)  # shape: (128, 128, 128), in z-y-x order
    return array


from monai.transforms import LoadImage, Compose, Spacing, Resize, EnsureChannelFirst, ScaleIntensity
import numpy as np

def monai_resample_to_128(input_path):
    transforms = Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),                   # shape: (1, H, W, D)
        ScaleIntensity(),                       # optional: normalize to 0-1
        Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"),  # resample to 1mm spacing
        Resize(spatial_size=(128, 128, 128), mode='trilinear')  # resize to 128³
    ])
    
    image = transforms(input_path)              # returns torch.Tensor
    array = image.numpy()[0]                    # remove channel dim → shape: (128, 128, 128)
    return array


In [74]:
for i in range(len(Files)):
    filename = Files[i]
    file_path = os.path.join(IXI_T1, filename)
    # img = nib.load(file_path)
    # out = img.get_fdata()
    out = monai_resample_to_128(file_path)
    print(np.unique(out)[-1])
    # out = resample_to_128(file_path)
    # for j in range(out.shape[1]):
    #     image = out[j, :, :]
    #     # print(np.unique(image))
        
    #     # Plotting the 2D array as an image
    #     plt.imshow(image)  # Use 'gray' for grayscale or try 'viridis', 'hot', etc.
    #     plt.colorbar()  # Optional: adds a color scale bar
    #     plt.title('2D Array Image')
    #     plt.axis('off')  # Hide axis ticks
    #     plt.show()
        
    # print(out.shape)

0.7694084
0.9390049
0.8991237
0.90730065
0.895421
0.7208875
0.8581001
0.7387637
0.9330733
0.8930943
0.9107981
0.90283346
0.81863487
0.82957613
0.8726671
0.93891776
0.9629158


In [80]:
pwd

'C:\\Users\\Rishabh\\Documents\\TransBTS'

In [86]:
import pandas as pd
CSV_Path = os.path.join(r'C:\Users\Rishabh\Documents\TransBTS', 'IXI.xlsx')
df = pd.read_excel(CSV_Path, engine='openpyxl')
# !pwd

In [87]:
df.head(10)

,IXI_ID,"SEX_ID (1=m, 2=f)",HEIGHT,WEIGHT,ETHNIC_ID,MARITAL_ID,OCCUPATION_ID,QUALIFICATION_ID,DOB,DATE_AVAILABLE,STUDY_DATE,AGE
0,1,1,170,80,2,3,5,2,1968-02-22,0,NaT,NaN
1,2,2,164,58,1,4,1,5,1970-01-30,1,2005-11-18,35.800137
2,12,1,175,70,1,2,1,5,1966-08-20,1,2005-06-01,38.781656
3,13,1,182,70,1,2,1,5,1958-09-15,1,2005-06-01,46.710472
4,14,2,163,65,1,4,1,5,1971-03-15,1,2005-06-09,34.236824
5,15,1,181,90,2,1,6,5,1981-03-11,1,2005-06-23,24.284736
6,16,1,172,63,1,2,1,5,1950-04-24,1,2005-06-24,55.167693
7,17,2,178,72,1,3,1,5,1976-05-21,1,2005-06-24,29.092402
8,19,1,180,88,1,2,1,4,1946-11-02,1,2005-06-30,58.658453
9,20,1,178,72,1,2,1,4,1966-01-05,1,2005-06-24,39.466119


In [88]:
df['IXI_ID']

0        1
1        2
2       12
3       13
4       14
      ... 
614    652
615    653
616    655
617    660
618    662
Name: IXI_ID, Length: 619, dtype: int64

In [105]:
for ID in ID_To_Filename.keys():
    print(df[df['IXI_ID'] == ID]['AGE'].values[0],'   ',ID_To_Filename[ID]  )
    

58.65845311430527     IXI019-Guys-0702-T1.nii.gz
26.272416153319643     IXI069-Guys-0769-T1.nii.gz
37.765913757700204     IXI110-Guys-0733-T1.nii.gz
27.014373716632445     IXI150-HH-1550-T1.nii.gz
39.397672826830934     IXI160-HH-1637-T1.nii.gz
48.05201916495551     IXI163-HH-1621-T1.nii.gz
67.53456536618754     IXI167-HH-1569-T1.nii.gz
25.713894592744694     IXI236-Guys-0817-T1.nii.gz
27.381245722108144     IXI265-Guys-0845-T1.nii.gz
73.56331279945243     IXI327-HH-1999-T1.nii.gz
67.77275838466804     IXI336-Guys-0904-T1.nii.gz
60.91991786447639     IXI363-HH-2050-T1.nii.gz
73.54414784394251     IXI399-Guys-0966-T1.nii.gz
59.742642026009584     IXI414-Guys-0957-T1.nii.gz
55.540041067761805     IXI429-Guys-0997-T1.nii.gz
28.347707049965777     IXI436-HH-2153-T1.nii.gz
46.42573579739904     IXI477-IOP-1141-T1.nii.gz


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

index = 15

# print(np.unique(outputarray))
# Example 2D array (you can replace this with your own array)
image = outputarray[index,:,:]
print(np.unique(image))

# Plotting the 2D array as an image
plt.imshow(image, cmap='gray')  # Use 'gray' for grayscale or try 'viridis', 'hot', etc.
plt.colorbar()  # Optional: adds a color scale bar
plt.title('2D Array Image')
plt.axis('off')  # Hide axis ticks
plt.show()